In [1]:
#%%
%load_ext autoreload
%autoreload 2

In [7]:
import os
from pathlib import Path

from monoculture.analysis.setup import ACS_TASKS, LLM_MODELS, prompt_connectors, prompt_styles
from monoculture.analysis.utils import model_to_key, is_instruction_tuned, load_json, find_files, parse_results_dict, create_result_df
import pandas as pd

RESULTS_ROOT_DIR = Path("./results/folktexts/")
subfolders = ['0-bullet-is', '0-bullet-colon', '0-bullet-equal', '0-text-is'] #, 'few-shot']

#### smaller version for one prompting style

In [8]:
model_folders = [dir+'/'+dir.replace('model-','')+f'_task-{task}' for task in ACS_TASKS for folder in subfolders for dir in os.listdir(RESULTS_ROOT_DIR/folder) if dir.startswith('model')]
cond_finished = lambda files: files and any([f.endswith('.json') for f in files]) and any([f.endswith('predictions.csv') for f in files])
finished_benches = [path for fold in model_folders for sub in subfolders for path,_,file_list in os.walk(RESULTS_ROOT_DIR/sub/fold) if cond_finished(file_list)]
finished_benches[:3], len(finished_benches)

(['results/folktexts/0-bullet-is/model-meta-llama--Meta-Llama-3.2-1B-Instruct/meta-llama--Meta-Llama-3.2-1B-Instruct_task-ACSIncome/meta-llama--Meta-Llama-3.2-1B-Instruct_bench-862320328',
  'results/folktexts/0-bullet-is/model-allenai--OLMo-2-1124-7B-Instruct/allenai--OLMo-2-1124-7B-Instruct_task-ACSIncome/allenai--OLMo-2-1124-7B-Instruct_bench-1231502606',
  'results/folktexts/0-bullet-is/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-3535356034'],
 1234)

In [9]:
model_folders

['model-meta-llama--Meta-Llama-3.2-1B-Instruct/meta-llama--Meta-Llama-3.2-1B-Instruct_task-ACSIncome',
 'model-allenai--OLMo-2-1124-7B-Instruct/allenai--OLMo-2-1124-7B-Instruct_task-ACSIncome',
 'model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome',
 'model-google--gemma-2-27b-it/google--gemma-2-27b-it_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3-70B/meta-llama--Meta-Llama-3-70B_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3-8B/meta-llama--Meta-Llama-3-8B_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3.1-70B-Instruct/meta-llama--Meta-Llama-3.1-70B-Instruct_task-ACSIncome',
 'model-google--gemma-2-9b/google--gemma-2-9b_task-ACSIncome',
 'model-allenai--OLMo-2-1124-7B/allenai--OLMo-2-1124-7B_task-ACSIncome',
 'model-Qwen--Qwen2-1.5B/Qwen--Qwen2-1.5B_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3-8B-Instruct/meta-llama--Meta-Llama-3-8B-Instruct_task-ACSIncome',
 'model-Qwen--Qwen2-72B-Instruct/Qwen--Qwen2-72B-Instruct_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3

In [10]:
finished_benches

['results/folktexts/0-bullet-is/model-meta-llama--Meta-Llama-3.2-1B-Instruct/meta-llama--Meta-Llama-3.2-1B-Instruct_task-ACSIncome/meta-llama--Meta-Llama-3.2-1B-Instruct_bench-862320328',
 'results/folktexts/0-bullet-is/model-allenai--OLMo-2-1124-7B-Instruct/allenai--OLMo-2-1124-7B-Instruct_task-ACSIncome/allenai--OLMo-2-1124-7B-Instruct_bench-1231502606',
 'results/folktexts/0-bullet-is/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-3535356034',
 'results/folktexts/0-bullet-colon/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-3440500242',
 'results/folktexts/0-bullet-equal/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-4235015672',
 'results/folktexts/0-text-is/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-54056871',
 'results/folktexts/0-bullet-is/model-google--gemma-2-27b-it/google--gemma-2-27b-it_task-ACSIncome/google--gemm

In [11]:
model_task_pairs = []
for folder_name in finished_benches:
    (model, task) = folder_name.split('/')[-2].split('_task-')
    model = model.replace('model-', '')
    model_task_pairs.append((model, task))
unique_model_task_pairs = list(set(model_task_pairs))
unique_model_task_pairs[0]

('meta-llama--Meta-Llama-3-8B-Instruct', 'ACSEmployment')

In [12]:
unique_model_task_pairs

[('meta-llama--Meta-Llama-3-8B-Instruct', 'ACSEmployment'),
 ('meta-llama--Meta-Llama-3-8B', 'ACSEmployment'),
 ('Qwen--Qwen2-72B', 'ACSEmployment'),
 ('google--gemma-2-9b', 'ACSIncome'),
 ('google--gemma-2-27b', 'ACSIncome'),
 ('google--gemma-2-9b-it', 'ACSIncome'),
 ('meta-llama--Meta-Llama-3.1-70B-Instruct', 'ACSEmployment'),
 ('allenai--OLMo-7B-hf', 'ACSEmployment'),
 ('meta-llama--Meta-Llama-3-70B', 'ACSPublicCoverage'),
 ('google--gemma-2b', 'ACSPublicCoverage'),
 ('allenai--OLMo-1B-0724-hf', 'ACSIncome'),
 ('meta-llama--Meta-Llama-3.2-3B', 'ACSEmployment'),
 ('01-ai--Yi-34B', 'ACSEmployment'),
 ('Qwen--Qwen2-7B-Instruct', 'ACSPublicCoverage'),
 ('mistralai--Mixtral-8x7B-v0.1', 'ACSPublicCoverage'),
 ('Qwen--Qwen2-7B', 'ACSIncome'),
 ('meta-llama--Meta-Llama-3-8B-Instruct', 'ACSPublicCoverage'),
 ('meta-llama--Meta-Llama-3-8B', 'ACSPublicCoverage'),
 ('Qwen--Qwen2-72B', 'ACSPublicCoverage'),
 ('allenai--OLMo-7B-Instruct-hf', 'ACSIncome'),
 ('mistralai--Mixtral-8x22B-Instruct-v0.1

In [13]:
show_available = False
show_unavailable = True
for task in ACS_TASKS:
    print(task)
    for model in LLM_MODELS:
        if (model_to_key(model), task) in unique_model_task_pairs:
            if show_available:
                print(f"- {model_to_key(model)}")
        else:
            if show_unavailable:
                print(f'x {model_to_key(model)}')
    print()

ACSIncome

ACSEmployment
x Qwen--Qwen2-72B-Instruct

ACSTravelTime
x Qwen--Qwen2-72B-Instruct

ACSPublicCoverage



### Check availability for different prompting styles

In [14]:
save_file_path = RESULTS_ROOT_DIR/'overview_results_by_prompt_style.csv'

In [15]:
load_df = False
df = pd.read_csv(save_file_path) if load_df else create_result_df(RESULTS_ROOT_DIR, subfolders=subfolders, tasks=ACS_TASKS, save_path=save_file_path)

(379, 9)


In [16]:
print(df.shape)
df[(df['num_shots']==0) & (df['task']!='ACSTravelTime')].head()
#zero-shot: 304 results - 19 models - 4 tasks

(379, 9)


,task,model,is_inst,bench_hash,num_shots,prompt_style,prompt_connector,eval_results_path,predictions_path
0,ACSIncome,meta-llama--Meta-Llama-3.2-1B-Instruct,1,862320328,0,bullet,is,results/folktexts/0-bullet-is/model-meta-llama...,results/0-bullet-is/model-meta-llama--Meta-Lla...
1,ACSIncome,allenai--OLMo-2-1124-7B-Instruct,1,1231502606,0,bullet,is,results/folktexts/0-bullet-is/model-allenai--O...,results/0-bullet-is/model-allenai--OLMo-2-1124...
2,ACSIncome,google--gemma-2-27b,0,3535356034,0,bullet,is,results/folktexts/0-bullet-is/model-google--ge...,results/0-bullet-is/model-google--gemma-2-27b/...
3,ACSIncome,google--gemma-2-27b-it,1,1373954821,0,bullet,is,results/folktexts/0-bullet-is/model-google--ge...,results/0-bullet-is/model-google--gemma-2-27b-...
4,ACSIncome,meta-llama--Meta-Llama-3-70B,0,730840778,0,bullet,is,results/folktexts/0-bullet-is/model-meta-llama...,results/0-bullet-is/model-meta-llama--Meta-Lla...


In [17]:
show_available = False
show_unavailable = True

for task_name in ACS_TASKS:
      print(task_name)
      for num_shots in [0]: ## only check zero-shot for now
            for sty in ['bullet']:
                  for con in ['is']:
                        if sty=='text' and con != 'is':
                                    continue
                        else:
                               print(f"  {num_shots} {sty} {con}")
                        for m in LLM_MODELS:
                              num_entries = df[(df['task']==task_name) & (df['model']==model_to_key(m)) & (df['prompt_style']==sty) & (df['prompt_connector']==con) & (df['num_shots'] == num_shots)].shape[0]
                              if show_available:
                                    if num_entries==1:
                                            print(f"\t- {m} ")
                                    elif num_entries > 1:
                                           print(f"\t- {m} -- Found multiple models with given characteristics.")
                              if show_unavailable and num_entries == 0:
                                    print(f"\tx {m}")
                              
                              


ACSIncome
  0 bullet is
ACSEmployment
  0 bullet is
	x Qwen/Qwen2-72B-Instruct
ACSTravelTime
  0 bullet is
	x Qwen/Qwen2-72B-Instruct
ACSPublicCoverage
  0 bullet is
